# Maroczy quickstart

End-to-end tour: connect to a running IBKR session, pull data, run the research features (bubble detection, covariance cleaning, CPCV, rough volatility), generate characteristics, build a signal, and backtest it.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd

from maroczy.broker import Broker, MarketData
from maroczy.features import bubble, covariance, cpcv, rough_vol
from maroczy.characteristics import CharacteristicEngine, CharacteristicRegistry
from maroczy.strategy import zscore, long_short_weights, run_backtest

## 1. Connect to IBKR

Requires TWS or IB Gateway running locally with the API enabled (File > Global Configuration > API > Settings). Defaults to the paper-trading port `7497`.

In [ ]:
broker = Broker()  # host='127.0.0.1', port=7497 (paper), client_id=1
broker.connect()
broker

In [ ]:
md = MarketData(broker)
symbols = ["AAPL", "MSFT", "AMZN", "GOOGL", "META"]
bars = {s: md.history(s, duration="3 Y", bar_size="1 day") for s in symbols}
bars["AAPL"].tail()

## 1b. London Strategic Edge (LSE) data + the IBKR-vs-LSE protocol

`maroczy.datafeed.UnifiedMarketData` gives you one `.history()` call that automatically
routes each request to IBKR or LSE. Rule of thumb: **IBKR** for real-time/streaming and
anything you're about to trade against; **LSE** for deep-history research pulls (no pacing
limits, cheaper, covers fundamentals/economics/options too). See the `maroczy.datafeed.router`
module docstring for the full decision procedure, summarized in the README.

Requires `pip install "maroczy[lse]"` and an `LSE_API_KEY`. Copy `.env.example` to `.env`
and fill it in once -- `import maroczy` loads it automatically, so `LSEData()` below just works.


In [ ]:
from maroczy.datafeed import LSEData, UnifiedMarketData, DataSourcePolicy

lse = LSEData()  # reads LSE_API_KEY from the environment
router = UnifiedMarketData(ibkr=md, lse=lse, policy=DataSourcePolicy(ibkr_intraday_limit_days=30))

# the protocol decides for you (see maroczy.datafeed.router docstring for the rules) ...
print("5y daily history       ->", router.resolve_source(bar_size="1 day", duration="5 Y"))
print("5-day 1-min history     ->", router.resolve_source(bar_size="1 min", duration="5 D"))
print("1-year 1-min history    ->", router.resolve_source(bar_size="1 min", duration="1 Y"))
print("real-time quote         ->", router.resolve_source(realtime=True))

# ... or you can force one explicitly
deep_history = router.history("AAPL", bar_size="1 day", duration="10 Y", source="lse")
deep_history.tail()


In [ ]:
# LSE also covers fundamentals/economics/options that IBKR doesn't expose cleanly --
# feed these into maroczy.characteristics.functions.funda / fundq.
income_stmt = lse.financial_reports("AAPL", report_type="income", period="FY")
fundamentals = lse.fundamentals("AAPL")
cpi_yoy = lse.economics("cpi_yoy")
us10y = lse.bond_yields("US10Y", start="2015-01-01")
income_stmt.head()


## 2. Bubble detection (Phillips, Shi & Yu, 2015)

In [ ]:
prices = bars["AAPL"]["close"]

result = bubble.gsadf(prices)
cv = bubble.critical_values(len(prices))
print(f"GSADF stat = {result.stat:.2f}  (95% critical value = {cv['gsadf'][0.95]:.2f})")

episodes = bubble.bsadf_dating(prices, quantile=0.95)
episodes

In [ ]:
ax = result.bsadf.plot(figsize=(10, 4), label="BSADF")
ax.axhline(cv["gsadf"][0.95], color="red", linestyle="--", label="95% critical value")
ax.legend();

## 3. Covariance matrix cleaning (Bongiorno & Challet, 2022)

In [ ]:
rets = pd.DataFrame({s: b["close"].pct_change() for s, b in bars.items()}).dropna()

raw_corr = rets.corr()
cleaned = covariance.clean_covariance(rets, method="cv")
print("condition number, raw:    ", covariance.condition_number(raw_corr))
print("condition number, cleaned:", covariance.condition_number(cleaned))

## 4. Combinatorial Purged Cross-Validation

In [ ]:
cv_splitter = cpcv.CombinatorialPurgedCV(n_groups=6, n_test_groups=2, embargo_frac=0.02)
print(f"{cv_splitter.n_splits} splits, {cv_splitter.n_paths} reconstructable backtest paths")

for split in list(cv_splitter.split(len(rets)))[:3]:
    print(split.test_groups, len(split.train_idx), len(split.test_idx))

## 5. Rough log-normal volatility (Hager, Horst, Wagenhofer & Xu, 2026)

In [ ]:
log_ret = np.log(bars["AAPL"]["close"]).diff().dropna()
realized_log_vol = np.log(log_ret.rolling(21).std().dropna())

H_hat, diagnostics = rough_vol.estimate_hurst(realized_log_vol)
print(f"Estimated Hurst exponent: {H_hat:.3f}")
print(f"Theoretical weak-convergence rate for microstructure MC pricing: {rough_vol.theoretical_weak_rate(max(H_hat, 0.01)):.3f}")

In [ ]:
sim = rough_vol.simulate_rough_bergomi(n_steps=252, n_paths=1000, H=max(H_hat, 0.05), eta=1.5, rho=-0.7, xi0=realized_log_vol.var())
sim["S"][:, -1].mean(), sim["S"][:, -1].std()

## 6. Characteristic generation from your 300+ characteristic CSV

In [ ]:
registry = CharacteristicRegistry()
print(registry)
print(registry.coverage())
registry.names(data_class="crspd")[:15]

In [ ]:
engine = CharacteristicEngine(registry)
mkt_ret = rets.mean(axis=1)  # equal-weight proxy for the market portfolio

panel = engine.compute_universe(
    bars,
    names=["ret_12_1", "ret_1_0", "retvol", "ami_126d", "ivol_capm_21d", "bidaskhl_21d"],
    mkt_ret=mkt_ret,
)
panel.tail(10)

## 7. Build a signal & backtest it

In [ ]:
mom = panel["ret_12_1"].unstack("symbol")
signal_z = zscore(mom, axis=1)

weights = signal_z.apply(lambda row: long_short_weights(row, n_long=2, n_short=2), axis=1)
returns = pd.DataFrame({s: b["close"].pct_change() for s, b in bars.items()})

result = run_backtest(weights, returns, cost_bps=5.0)
result

In [ ]:
result.equity_curve.plot(figsize=(10, 4), title="Momentum long/short equity curve");

## 8. (Optional) Live signal loop

Dry-run by default (`auto=False`) -- inspect `.pending_orders` before ever setting `auto=True`.

In [ ]:
from maroczy.broker import LiveExecutor

def my_signal(symbol: str) -> float | None:
    # plug in your latest cross-sectional signal lookup here
    return signal_z.iloc[-1].get(symbol)

executor = LiveExecutor(broker=broker, signal_fn=my_signal, capital=10_000, auto=False)
executor.run(symbols)